In [5]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import Normalizer
import joblib

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

from channel_estimator import compute_channel_matrix_from_iq_paths
from channel_features_complete import extract_channel_matrix_features

# =============================================================================
# 2. FASE DE ENTRENAMIENTO (CON LA BBDD ANTIGUA)
# =============================================================================
print("🧠 1. Entrenando modelo con la BBDD Antigua (dataset_features_temperatura.csv)...")

# Cargar el CSV de Hugo
df_antiguo = pd.read_csv('../dataset_features_temperatura.csv')

# Descartamos la Fase (porque vimos que en el cartón es puro ruido 0.9999)
# Nos quedamos con las dinámicas de Doppler y SVD
variables_clave = [
    'doppler_variance_energy', 
    'dH_dt_mean', 
    'doppler_centroid', 
    'doppler_spread',
    'svd_sigma_ratio'
]

# Asegurar que existen en el CSV
variables_finales = [v for v in variables_clave if v in df_antiguo.columns]

X_train_bruto = df_antiguo[variables_finales]
y_train = df_antiguo['temperature']

# APLICAMOS LA MAGIA: Normalizador L2 por fila
normalizador = Normalizer(norm='l2')
X_train_norm = normalizador.fit_transform(X_train_bruto)

# Entrenamos el Random Forest con los datos aplastados a proporciones
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
modelo_rf.fit(X_train_norm, y_train)

print("✅ Modelo entrenado y normalizador configurado.")

# =============================================================================
# 3. FASE DE TESTEO EN VIVO (CON EL VASO DE CARTÓN)
# =============================================================================
print("\n📡 2. Procesando grabaciones del Vaso de Cartón...")
RUTA_YAML = Path("../../data/Modulator.yaml")
BBDD_DIR = Path("../Datos TEST Vaso carton")

pruebas = [
    {"nombre": "Vaso Frío - Muestra 1", "tx": BBDD_DIR/"frio"/"iq_tx_1.bin", "rx": BBDD_DIR/"frio"/"iq_rx_1.bin"},
    {"nombre": "Vaso Frío - Muestra 2", "tx": BBDD_DIR/"frio"/"iq_tx_2.bin", "rx": BBDD_DIR/"frio"/"iq_rx_2.bin"},
    {"nombre": "Vaso Templado - Muestra 1", "tx": BBDD_DIR/"templado"/"iq_tx_1.bin", "rx": BBDD_DIR/"templado"/"iq_rx_1.bin"},
    {"nombre": "Vaso Caliente - Muestra 1", "tx": BBDD_DIR/"caliente"/"iq_tx_1.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_1.bin"},
    {"nombre": "Vaso Caliente - Muestra 2", "tx": BBDD_DIR/"caliente"/"iq_tx_2.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_2.bin"}
]

print("-" * 50)
print("🎯 RESULTADOS FINALES DE PREDICCIÓN")
print("-" * 50)

for p in pruebas:
    # EL CHIVATO: Si no existen, que nos diga exactamente DÓNDE está buscando
    if not p["tx"].exists() or not p["rx"].exists():
        print(f"⚠️ Aviso: Saltando '{p['nombre']}'.")
        print(f"   -> Python lo está buscando en la ruta equivocada:")
        print(f"   -> {p['tx'].resolve()}")
        continue
        
    try:
        # Extraer características del radar
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], rx_path=p["rx"], yaml_path=RUTA_YAML,
            fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
            output_order="mk", verbose=False
        )
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        # Quedarnos solo con las variables clave
        features_filtradas = {var: todas_features[var] for var in variables_finales}
        df_test_bruto = pd.DataFrame([features_filtradas])
        
        # APLICAMOS LA MAGIA AL TEST: Pasamos los datos por el normalizador
        X_test_norm = normalizador.transform(df_test_bruto)
        
        # Predecimos la temperatura
        temp_detectada = modelo_rf.predict(X_test_norm)[0]
        
        print(f"🌡️ [{p['nombre']}] -> Predicción IA: {temp_detectada:.1f} ºC")
        
    except Exception as e:
        print(f"❌ Error interno en {p['nombre']}: {e}")

print("-" * 50)

🧠 1. Entrenando modelo con la BBDD Antigua (dataset_features_temperatura.csv)...
✅ Modelo entrenado y normalizador configurado.

📡 2. Procesando grabaciones del Vaso de Cartón...
--------------------------------------------------
🎯 RESULTADOS FINALES DE PREDICCIÓN
--------------------------------------------------
🌡️ [Vaso Frío - Muestra 1] -> Predicción IA: 40.0 ºC
🌡️ [Vaso Frío - Muestra 2] -> Predicción IA: 40.0 ºC
🌡️ [Vaso Templado - Muestra 1] -> Predicción IA: 40.0 ºC
🌡️ [Vaso Caliente - Muestra 1] -> Predicción IA: 40.0 ºC
🌡️ [Vaso Caliente - Muestra 2] -> Predicción IA: 40.0 ºC
--------------------------------------------------


In [9]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

from channel_estimator import compute_channel_matrix_from_iq_paths
from channel_features_complete import extract_channel_matrix_features

# =============================================================================
# 2. FASE DE ENTRENAMIENTO (UNIVERSO 1: BBDD ANTIGUA)
# =============================================================================
print("🧠 1. Entrenando modelo con la BBDD Antigua (dataset_features_temperatura.csv)...")

df_antiguo = pd.read_csv('../dataset_features_temperatura.csv')

variables_clave = [
    'doppler_variance_energy', 
    'dH_dt_mean', 
    'doppler_centroid', 
    'doppler_spread',
    'svd_sigma_ratio'
]

variables_finales = [v for v in variables_clave if v in df_antiguo.columns]

X_train_bruto = df_antiguo[variables_finales]
y_train = df_antiguo['temperature']

# ESCALADOR 1: Aprende los límites de la base de datos antigua (de 0 a 1)
scaler_train = MinMaxScaler()
X_train_scaled = scaler_train.fit_transform(X_train_bruto)

# Entrenamos el Random Forest
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
modelo_rf.fit(X_train_scaled, y_train)

print("✅ IA entrenada con éxito en el Universo Antiguo.")

# =============================================================================
# 3. EXTRACCIÓN DE DATOS EN VIVO (UNIVERSO 2: VASO CARTÓN)
# =============================================================================
print("\n📡 2. Procesando grabaciones del Vaso de Cartón...")
RUTA_YAML = Path("../../data/Modulator.yaml")
BBDD_DIR = Path("../Datos TEST Vaso carton")

pruebas = [
    {"nombre": "Vaso Frío - Muestra 1", "tx": BBDD_DIR/"frio"/"iq_tx_1.bin", "rx": BBDD_DIR/"frio"/"iq_rx_1.bin"},
    {"nombre": "Vaso Frío - Muestra 2", "tx": BBDD_DIR/"frio"/"iq_tx_2.bin", "rx": BBDD_DIR/"frio"/"iq_rx_2.bin"},
    {"nombre": "Vaso Templado - Muestra 1", "tx": BBDD_DIR/"templado"/"iq_tx_1.bin", "rx": BBDD_DIR/"templado"/"iq_rx_1.bin"},
    {"nombre": "Vaso Templado - Muestra 2", "tx": BBDD_DIR/"templado"/"iq_tx_2.bin", "rx": BBDD_DIR/"templado"/"iq_rx_2.bin"},
    {"nombre": "Vaso Caliente - Muestra 1", "tx": BBDD_DIR/"caliente"/"iq_tx_1.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_1.bin"},
    {"nombre": "Vaso Caliente - Muestra 2", "tx": BBDD_DIR/"caliente"/"iq_tx_2.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_2.bin"}
]

datos_test_lista = []
nombres_validos = []

for p in pruebas:
    if not p["tx"].exists() or not p["rx"].exists():
        continue
        
    try:
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], rx_path=p["rx"], yaml_path=RUTA_YAML,
            fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
            output_order="mk", verbose=False
        )
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        features_filtradas = {var: todas_features[var] for var in variables_finales}
        datos_test_lista.append(features_filtradas)
        nombres_validos.append(p["nombre"])
        
    except Exception as e:
        print(f"❌ Error en {p['nombre']}: {e}")

# =============================================================================
# 4. ADAPTACIÓN DE DOMINIO Y PREDICCIÓN
# =============================================================================
print("-" * 50)
print("🎯 RESULTADOS FINALES DE PREDICCIÓN")
print("-" * 50)

if len(datos_test_lista) > 0:
    df_test_bruto = pd.DataFrame(datos_test_lista)
    
    # ESCALADOR 2: Aprende los límites de la prueba de hoy (de 0 a 1)
    # Esto fuerza al modelo a juzgar los vasos comparándolos entre sí.
    scaler_test = MinMaxScaler()
    # Añadimos un '-' para invertir la física del cartón y alinearla con la BBDD antigua
    X_test_scaled = scaler_test.fit_transform(-df_test_bruto)
    
    # Predecir
    predicciones = modelo_rf.predict(X_test_scaled)
    
    for nombre, temp in zip(nombres_validos, predicciones):
        print(f"🌡️ [{nombre}] -> Predicción IA: {temp:.1f} ºC")
else:
    print("❌ No se ha podido procesar ningún archivo de test.")

print("-" * 50)

🧠 1. Entrenando modelo con la BBDD Antigua (dataset_features_temperatura.csv)...
✅ IA entrenada con éxito en el Universo Antiguo.

📡 2. Procesando grabaciones del Vaso de Cartón...
--------------------------------------------------
🎯 RESULTADOS FINALES DE PREDICCIÓN
--------------------------------------------------
🌡️ [Vaso Frío - Muestra 1] -> Predicción IA: 23.0 ºC
🌡️ [Vaso Frío - Muestra 2] -> Predicción IA: 23.0 ºC
🌡️ [Vaso Templado - Muestra 1] -> Predicción IA: 23.0 ºC
🌡️ [Vaso Templado - Muestra 2] -> Predicción IA: 80.0 ºC
🌡️ [Vaso Caliente - Muestra 1] -> Predicción IA: 37.2 ºC
🌡️ [Vaso Caliente - Muestra 2] -> Predicción IA: 42.2 ºC
--------------------------------------------------


In [12]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler # 1. CAMBIADO A STANDARDSCALER

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

from channel_estimator import compute_channel_matrix_from_iq_paths
from channel_features_complete import extract_channel_matrix_features

# =============================================================================
# 2. FASE DE ENTRENAMIENTO (UNIVERSO 1: BBDD ANTIGUA)
# =============================================================================
print("🧠 1. Entrenando modelo con la BBDD Antigua (dataset_features_temperatura.csv)...")

df_antiguo = pd.read_csv('../dataset_features_temperatura.csv')

variables_clave = [
    'doppler_variance_energy', 
    'dH_dt_mean', 
    'doppler_centroid', 
    'doppler_spread',
    'svd_sigma_ratio'
]

variables_finales = [v for v in variables_clave if v in df_antiguo.columns]

X_train_bruto = df_antiguo[variables_finales]
y_train = df_antiguo['temperature']

# ESCALADOR 1: StandardScaler para el entrenamiento (centrado en la media)
scaler_train = StandardScaler()
X_train_scaled = scaler_train.fit_transform(X_train_bruto)

# Entrenamos el Random Forest
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
modelo_rf.fit(X_train_scaled, y_train)

print("✅ IA entrenada con éxito en el Universo Antiguo.")

# =============================================================================
# 3. EXTRACCIÓN DE DATOS EN VIVO (UNIVERSO 2: VASO CARTÓN)
# =============================================================================
print("\n📡 2. Procesando grabaciones del Vaso de Cartón...")
RUTA_YAML = Path("../../data/Modulator.yaml")
BBDD_DIR = Path("../Datos TEST Vaso carton")

pruebas = [
    {"nombre": "Vaso Frío - Muestra 1", "tx": BBDD_DIR/"frio"/"iq_tx_1.bin", "rx": BBDD_DIR/"frio"/"iq_rx_1.bin"},
    {"nombre": "Vaso Frío - Muestra 2", "tx": BBDD_DIR/"frio"/"iq_tx_2.bin", "rx": BBDD_DIR/"frio"/"iq_rx_2.bin"},
    {"nombre": "Vaso Templado - Muestra 1", "tx": BBDD_DIR/"templado"/"iq_tx_1.bin", "rx": BBDD_DIR/"templado"/"iq_rx_1.bin"},
    {"nombre": "Vaso Templado - Muestra 2", "tx": BBDD_DIR/"templado"/"iq_tx_2.bin", "rx": BBDD_DIR/"templado"/"iq_rx_2.bin"},
    {"nombre": "Vaso Caliente - Muestra 1", "tx": BBDD_DIR/"caliente"/"iq_tx_1.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_1.bin"},
    {"nombre": "Vaso Caliente - Muestra 2", "tx": BBDD_DIR/"caliente"/"iq_tx_2.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_2.bin"}
]

datos_test_lista = []
nombres_validos = []

for p in pruebas:
    if not p["tx"].exists() or not p["rx"].exists():
        continue
        
    try:
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], rx_path=p["rx"], yaml_path=RUTA_YAML,
            fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
            output_order="mk", verbose=False
        )
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        features_filtradas = {var: todas_features[var] for var in variables_finales}
        datos_test_lista.append(features_filtradas)
        nombres_validos.append(p["nombre"])
        
    except Exception as e:
        print(f"❌ Error en {p['nombre']}: {e}")

# =============================================================================
# 4. ADAPTACIÓN DE DOMINIO Y PREDICCIÓN
# =============================================================================
print("-" * 50)
print("🎯 RESULTADOS FINALES DE PREDICCIÓN")
print("-" * 50)

if len(datos_test_lista) > 0:
    df_test_bruto = pd.DataFrame(datos_test_lista)
    
    # 2. CHIVATO: Imprimir valores raw para analizar el "outlier"
    print("\n🔍 VALORES RAW (Antes de escalar):")
    df_mostrar = df_test_bruto.copy()
    df_mostrar.insert(0, 'Nombre', nombres_validos)
    print(df_mostrar.to_string(index=False))
    print("\n")
    
    # ESCALADOR 2: StandardScaler aísla los valores atípicos protegiendo a los demás
    scaler_test = StandardScaler()
    
    # 3. Mantenemos el '-' para invertir la física del cartón y alinearla con la BBDD antigua
    X_test_scaled = scaler_test.fit_transform(df_test_bruto)
    
    # Predecir
    predicciones = modelo_rf.predict(X_test_scaled)
    
    for nombre, temp in zip(nombres_validos, predicciones):
        print(f"🌡️ [{nombre}] -> Predicción IA: {temp:.1f} ºC")
else:
    print("❌ No se ha podido procesar ningún archivo de test.")

print("-" * 50)

🧠 1. Entrenando modelo con la BBDD Antigua (dataset_features_temperatura.csv)...
✅ IA entrenada con éxito en el Universo Antiguo.

📡 2. Procesando grabaciones del Vaso de Cartón...
--------------------------------------------------
🎯 RESULTADOS FINALES DE PREDICCIÓN
--------------------------------------------------

🔍 VALORES RAW (Antes de escalar):
                   Nombre  doppler_variance_energy  dH_dt_mean  doppler_centroid  doppler_spread  svd_sigma_ratio
    Vaso Frío - Muestra 1                 2.348526    0.000473          0.251797        0.289564        87.178024
    Vaso Frío - Muestra 2                 2.553152    0.000516          0.252320        0.289995        93.976201
Vaso Templado - Muestra 1                 3.210892    0.000641          0.250256        0.288263       121.162407
Vaso Templado - Muestra 2                 3.433427    0.000690          0.251381        0.289163       113.566843
Vaso Caliente - Muestra 1                 7.423988    0.001491          0.251

In [13]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler # 1. CAMBIADO A STANDARDSCALER

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

from channel_estimator import compute_channel_matrix_from_iq_paths
from channel_features_complete import extract_channel_matrix_features

# =============================================================================
# 2. FASE DE ENTRENAMIENTO (UNIVERSO 1: BBDD ANTIGUA)
# =============================================================================
print("🧠 1. Entrenando modelo con la BBDD Antigua (dataset_features_temperatura.csv)...")

df_antiguo = pd.read_csv('../dataset_features_temperatura.csv')

variables_clave = [
    'doppler_variance_energy', 
    'dH_dt_mean', 
    'svd_sigma_ratio'
]

variables_finales = [v for v in variables_clave if v in df_antiguo.columns]

X_train_bruto = df_antiguo[variables_finales]
y_train = df_antiguo['temperature']

# ESCALADOR 1: StandardScaler para el entrenamiento (centrado en la media)
scaler_train = StandardScaler()
X_train_scaled = scaler_train.fit_transform(X_train_bruto)

# Entrenamos el Random Forest
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
modelo_rf.fit(X_train_scaled, y_train)

print("✅ IA entrenada con éxito en el Universo Antiguo.")

# =============================================================================
# 3. EXTRACCIÓN DE DATOS EN VIVO (UNIVERSO 2: VASO CARTÓN)
# =============================================================================
print("\n📡 2. Procesando grabaciones del Vaso de Cartón...")
RUTA_YAML = Path("../../data/Modulator.yaml")
BBDD_DIR = Path("../Datos TEST Vaso carton")

pruebas = [
    {"nombre": "Vaso Frío - Muestra 1", "tx": BBDD_DIR/"frio"/"iq_tx_1.bin", "rx": BBDD_DIR/"frio"/"iq_rx_1.bin"},
    {"nombre": "Vaso Frío - Muestra 2", "tx": BBDD_DIR/"frio"/"iq_tx_2.bin", "rx": BBDD_DIR/"frio"/"iq_rx_2.bin"},
    {"nombre": "Vaso Templado - Muestra 1", "tx": BBDD_DIR/"templado"/"iq_tx_1.bin", "rx": BBDD_DIR/"templado"/"iq_rx_1.bin"},
    {"nombre": "Vaso Templado - Muestra 2", "tx": BBDD_DIR/"templado"/"iq_tx_2.bin", "rx": BBDD_DIR/"templado"/"iq_rx_2.bin"},
    {"nombre": "Vaso Caliente - Muestra 1", "tx": BBDD_DIR/"caliente"/"iq_tx_1.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_1.bin"},
    {"nombre": "Vaso Caliente - Muestra 2", "tx": BBDD_DIR/"caliente"/"iq_tx_2.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_2.bin"}
]

datos_test_lista = []
nombres_validos = []

for p in pruebas:
    if not p["tx"].exists() or not p["rx"].exists():
        continue
        
    try:
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], rx_path=p["rx"], yaml_path=RUTA_YAML,
            fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
            output_order="mk", verbose=False
        )
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        features_filtradas = {var: todas_features[var] for var in variables_finales}
        datos_test_lista.append(features_filtradas)
        nombres_validos.append(p["nombre"])
        
    except Exception as e:
        print(f"❌ Error en {p['nombre']}: {e}")

# =============================================================================
# 4. ADAPTACIÓN DE DOMINIO Y PREDICCIÓN
# =============================================================================
print("-" * 50)
print("🎯 RESULTADOS FINALES DE PREDICCIÓN")
print("-" * 50)

if len(datos_test_lista) > 0:
    df_test_bruto = pd.DataFrame(datos_test_lista)
    
    # 2. CHIVATO: Imprimir valores raw para analizar el "outlier"
    print("\n🔍 VALORES RAW (Antes de escalar):")
    df_mostrar = df_test_bruto.copy()
    df_mostrar.insert(0, 'Nombre', nombres_validos)
    print(df_mostrar.to_string(index=False))
    print("\n")
    
    # ESCALADOR 2: StandardScaler aísla los valores atípicos protegiendo a los demás
    scaler_test = StandardScaler()
    
    # 3. Mantenemos el '-' para invertir la física del cartón y alinearla con la BBDD antigua
    X_test_scaled = scaler_test.fit_transform(df_test_bruto)
    
    # Predecir
    predicciones = modelo_rf.predict(X_test_scaled)
    
    for nombre, temp in zip(nombres_validos, predicciones):
        print(f"🌡️ [{nombre}] -> Predicción IA: {temp:.1f} ºC")
else:
    print("❌ No se ha podido procesar ningún archivo de test.")

print("-" * 50)

🧠 1. Entrenando modelo con la BBDD Antigua (dataset_features_temperatura.csv)...
✅ IA entrenada con éxito en el Universo Antiguo.

📡 2. Procesando grabaciones del Vaso de Cartón...
--------------------------------------------------
🎯 RESULTADOS FINALES DE PREDICCIÓN
--------------------------------------------------

🔍 VALORES RAW (Antes de escalar):
                   Nombre  doppler_variance_energy  dH_dt_mean  svd_sigma_ratio
    Vaso Frío - Muestra 1                 2.348526    0.000473        87.178024
    Vaso Frío - Muestra 2                 2.553152    0.000516        93.976201
Vaso Templado - Muestra 1                 3.210892    0.000641       121.162407
Vaso Templado - Muestra 2                 3.433427    0.000690       113.566843
Vaso Caliente - Muestra 1                 7.423988    0.001491       182.607716
Vaso Caliente - Muestra 2                 7.348954    0.001478       178.148135


🌡️ [Vaso Frío - Muestra 1] -> Predicción IA: 47.9 ºC
🌡️ [Vaso Frío - Muestra 2] -> Pr

In [15]:
import sys
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler

# =============================================================================
# 1. CONECTAR FUNCIONES DE PROCESADO
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

# =============================================================================
# 2. FASE DE ENTRENAMIENTO (UNIVERSO 1: BBDD ANTIGUA)
# =============================================================================
print("🧠 1. Entrenando modelo con la BBDD Antigua (dataset_features_temperatura.csv)...")

df_antiguo = pd.read_csv('../dataset_features_temperatura.csv')

# Nos quedamos estrictamente con las 3 variables más limpias y sin ruido
variables_clave = [
    'doppler_variance_energy', 
    'dH_dt_mean', 
    'svd_sigma_ratio'
]

variables_finales = [v for v in variables_clave if v in df_antiguo.columns]

X_train_bruto = df_antiguo[variables_finales]
y_train = df_antiguo['temperature']

# ESCALADOR 1: Aprende los límites de la base de datos antigua (de 0 a 1)
scaler_train = MinMaxScaler()
X_train_scaled = scaler_train.fit_transform(X_train_bruto)

# Entrenamos el Random Forest
modelo_rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
modelo_rf.fit(X_train_scaled, y_train)

print("✅ IA entrenada con éxito en el Universo Antiguo.")

# =============================================================================
# 3. EXTRACCIÓN DE DATOS EN VIVO (UNIVERSO 2: VASO CARTÓN)
# =============================================================================
print("\n📡 2. Procesando grabaciones del Vaso de Cartón...")
RUTA_YAML = Path("../../data/Modulator.yaml")
BBDD_DIR = Path("../Datos TEST Vaso carton")

pruebas = [
    {"nombre": "Vaso Frío - Muestra 1", "tx": BBDD_DIR/"frio"/"iq_tx_1.bin", "rx": BBDD_DIR/"frio"/"iq_rx_1.bin"},
    {"nombre": "Vaso Frío - Muestra 2", "tx": BBDD_DIR/"frio"/"iq_tx_2.bin", "rx": BBDD_DIR/"frio"/"iq_rx_2.bin"},
    {"nombre": "Vaso Templado - Muestra 1", "tx": BBDD_DIR/"templado"/"iq_tx_1.bin", "rx": BBDD_DIR/"templado"/"iq_rx_1.bin"},
    {"nombre": "Vaso Templado - Muestra 2", "tx": BBDD_DIR/"templado"/"iq_tx_2.bin", "rx": BBDD_DIR/"templado"/"iq_rx_2.bin"},
    {"nombre": "Vaso Caliente - Muestra 1", "tx": BBDD_DIR/"caliente"/"iq_tx_1.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_1.bin"},
    {"nombre": "Vaso Caliente - Muestra 2", "tx": BBDD_DIR/"caliente"/"iq_tx_2.bin", "rx": BBDD_DIR/"caliente"/"iq_rx_2.bin"}
]

datos_test_lista = []
nombres_validos = []

for p in pruebas:
    if not p["tx"].exists() or not p["rx"].exists():
        continue
        
    try:
        H = compute_channel_matrix_from_iq_paths(
            tx_path=p["tx"], rx_path=p["rx"], yaml_path=RUTA_YAML,
            fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
            output_order="mk", verbose=False
        )
        todas_features = extract_channel_matrix_features(H, input_order="mk")
        
        # Filtramos usando solo las 3 variables clave
        features_filtradas = {var: todas_features[var] for var in variables_finales}
        datos_test_lista.append(features_filtradas)
        nombres_validos.append(p["nombre"])
        
    except Exception as e:
        print(f"❌ Error en {p['nombre']}: {e}")

# =============================================================================
# 4. ADAPTACIÓN DE DOMINIO Y PREDICCIÓN
# =============================================================================
print("-" * 50)
print("🎯 RESULTADOS FINALES DE PREDICCIÓN")
print("-" * 50)

if len(datos_test_lista) > 0:
    df_test_bruto = pd.DataFrame(datos_test_lista)
    
    # CHIVATO: Imprimir valores raw para verificar la progresión limpia
    print("\n🔍 VALORES RAW (Antes de escalar):")
    df_mostrar = df_test_bruto.copy()
    df_mostrar.insert(0, 'Nombre', nombres_validos)
    print(df_mostrar.to_string(index=False))
    print("\n")
    
    # ESCALADOR 2: Mapea la prueba en vivo de 0 a 1 (Sin el signo negativo)
    scaler_test = MinMaxScaler()
    X_test_scaled = scaler_test.fit_transform(df_test_bruto)
    
    # Predecir
    predicciones = modelo_rf.predict(X_test_scaled)
    
    for nombre, temp in zip(nombres_validos, predicciones):
        print(f"🌡️ [{nombre}] -> Predicción IA: {temp:.1f} ºC")
else:
    print("❌ No se ha podido procesar ningún archivo de test.")

print("-" * 50)

🧠 1. Entrenando modelo con la BBDD Antigua (dataset_features_temperatura.csv)...
✅ IA entrenada con éxito en el Universo Antiguo.

📡 2. Procesando grabaciones del Vaso de Cartón...
--------------------------------------------------
🎯 RESULTADOS FINALES DE PREDICCIÓN
--------------------------------------------------

🔍 VALORES RAW (Antes de escalar):
                   Nombre  doppler_variance_energy  dH_dt_mean  svd_sigma_ratio
    Vaso Frío - Muestra 1                 2.348526    0.000473        87.178024
    Vaso Frío - Muestra 2                 2.553152    0.000516        93.976201
Vaso Templado - Muestra 1                 3.210892    0.000641       121.162407
Vaso Templado - Muestra 2                 3.433427    0.000690       113.566843
Vaso Caliente - Muestra 1                 7.423988    0.001491       182.607716
Vaso Caliente - Muestra 2                 7.348954    0.001478       178.148135


🌡️ [Vaso Frío - Muestra 1] -> Predicción IA: 47.9 ºC
🌡️ [Vaso Frío - Muestra 2] -> Pr